# Data Wrangling

In this notebook, I continued on the analysis of the data collected on the game: Arc Raiders through YouTube comments.

In [7]:
import numpy as np
import pandas as pd
import re
from datetime import datetime, timedelta
from scipy import stats
import os

# For text preprocessing (for RoBERTa later)
import warnings
warnings.filterwarnings('ignore')

In [8]:
# Load the dataset
data = pd.read_csv('data/comments_data.csv')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 215433 entries, 0 to 215432
Data columns (total 34 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   comment_id         215433 non-null  str    
 1   text               215361 non-null  str    
 2   comment_date       215433 non-null  str    
 3   author_hash        215433 non-null  str    
 4   parent_id          48887 non-null   str    
 5   last_updated_at    215433 non-null  str    
 6   video_id           215433 non-null  str    
 7   likes              215433 non-null  int64  
 8   video_title        215433 non-null  str    
 9   video_date         215433 non-null  str    
 10  channel_id         215433 non-null  str    
 11  keyword_matched    215433 non-null  str    
 12  video_description  203656 non-null  str    
 13  char_count         215433 non-null  int64  
 14  word_count         215433 non-null  int64  
 15  avg_word_length    215433 non-null  float64
 16  has_url      

## 1. Data Cleaning

### 1.1. Handling Missing Values

In [9]:
data.isna().sum()

comment_id                0
text                     72
comment_date              0
author_hash               0
parent_id            166546
last_updated_at           0
video_id                  0
likes                     0
video_title               0
video_date                0
channel_id                0
keyword_matched           0
video_description     11777
char_count                0
word_count                0
avg_word_length           0
has_url                   0
has_mention               0
has_hashtag               0
exclamation_count         0
question_count            0
emoji_count               0
newline_count             0
uppercase_ratio           0
language                  0
day_of_week               0
day_num                   0
date                      0
is_spike_x                0
z_score_x              2064
is_spike_y                0
z_score_y              2064
is_spike                  0
z_score                2064
dtype: int64

In [10]:
data[data['video_description'].notna()]

,comment_id,text,comment_date,author_hash,parent_id,last_updated_at,video_id,likes,video_title,video_date,...,language,day_of_week,day_num,date,is_spike_x,z_score_x,is_spike_y,z_score_y,is_spike,z_score
0,UgxEgHTlwp52_t7tvld4AaABAg,this video is literally a lie - Arc Raiders do...,2026-03-12 09:00:00,00c95ebeeeb9b2094e02791c81e2ac9cd9e3a68724e2c5...,NaN,2026-03-12 09:00:00.000000,xuftkDxjGT4,1,Arc Raiders Reveal Trailer | Game Awards 2021,2021-12-10 03:58:14,...,en,Thursday,3,2026-03-12,False,-1.004165,False,-1.004165,False,-1.004165
1,UgxfQBIpu5Q9a2pqKD54AaABAg,Caraca não sabia que era tão antigo,2026-03-12 03:34:19,bef66a381d8737e126860408d372d8d4e8e463419de30a...,NaN,2026-03-12 03:34:19.000000,xuftkDxjGT4,0,Arc Raiders Reveal Trailer | Game Awards 2021,2021-12-10 03:58:14,...,pt,Thursday,3,2026-03-12,False,-1.004165,False,-1.004165,False,-1.004165
2,Ugx2KM9kJqjh_mesOIZ4AaABAg,😮This trailer is the greatest prequel lore in ...,2026-03-10 23:32:15,31c593066ff6110d558cc310a246c895d7bcf840d91e93...,NaN,2026-03-10 23:32:15.000000,xuftkDxjGT4,0,Arc Raiders Reveal Trailer | Game Awards 2021,2021-12-10 03:58:14,...,en,Tuesday,1,2026-03-10,False,1.332162,False,1.332162,False,1.332162
3,Ugyt0PrH_pOeXqKrYRN4AaABAg,holy false advertising,2026-03-10 17:56:56,8890a908eaecc345c51e5e6716db3766e3b4044fbd0434...,NaN,2026-03-10 17:56:56.000000,xuftkDxjGT4,2,Arc Raiders Reveal Trailer | Game Awards 2021,2021-12-10 03:58:14,...,no,Tuesday,1,2026-03-10,False,1.332162,False,1.332162,False,1.332162
4,UgxIu-puc-eEI2TVJ4h4AaABAg,Best game,2026-03-10 09:58:20,4ad74d9af134db04afdc8d76a0b60d2193b82d12ac29a3...,NaN,2026-03-10 09:58:20.000000,xuftkDxjGT4,0,Arc Raiders Reveal Trailer | Game Awards 2021,2021-12-10 03:58:14,...,de,Tuesday,1,2026-03-10,False,1.332162,False,1.332162,False,1.332162
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215174,UgyWix-PnxqpVonB2U94AaABAg,I agree with six. I'm cool as long as they're ...,2026-02-16 01:53:28,c05692a8fe9fbc022e732fcfc927652fc87d5bde5b23da...,NaN,2026-02-16 01:53:28.000000,6vm2rpXiykI,43,Arc Raiders Funny Moments,2026-02-16 01:44:28,...,en,Monday,0,2026-02-16,False,1.483043,False,1.483043,False,1.483043
215175,UgyWix-PnxqpVonB2U94AaABAg.ATGlzq_ZpuRATHCGtcOaHS,"also seven eight nine, that´s why he choose si...",2026-02-16 05:51:52,a94495c566b934a9d8a1737c9a1344010b824a2c7b0f47...,UgyWix-PnxqpVonB2U94AaABAg,2026-02-16 05:51:52.000000,6vm2rpXiykI,2,Arc Raiders Funny Moments,2026-02-16 01:44:28,...,en,Monday,0,2026-02-16,False,1.483043,False,1.483043,False,1.483043
215176,UgyWix-PnxqpVonB2U94AaABAg.ATGlzq_ZpuRATIkXerShOm,dude youtube surveyed your comment i closed it,2026-02-16 20:19:14,ae9b2006ee210b025ef236f674fb50e7c38e2a41fa2b73...,UgyWix-PnxqpVonB2U94AaABAg,2026-02-16 20:19:14.000000,6vm2rpXiykI,0,Arc Raiders Funny Moments,2026-02-16 01:44:28,...,en,Monday,0,2026-02-16,False,1.483043,False,1.483043,False,1.483043
215177,Ugx309ccJZu9k9Zao214AaABAg,Yes. This game is made for Adan,2026-02-16 01:50:34,0b606432fd11b715da9e56a7d6792a40f14391292ae91d...,NaN,2026-02-16 01:50:34.000000,6vm2rpXiykI,96,Arc Raiders Funny Moments,2026-02-16 01:44:28,...,en,Monday,0,2026-02-16,False,1.483043,False,1.483043,False,1.483043


In [11]:
# Filter out rows with missing video descriptions and comments
filtered_data = data[data['video_description'].notna() & data['text'].notna()]
filtered_data.info()

<class 'pandas.DataFrame'>
Index: 203585 entries, 0 to 215178
Data columns (total 34 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   comment_id         203585 non-null  str    
 1   text               203585 non-null  str    
 2   comment_date       203585 non-null  str    
 3   author_hash        203585 non-null  str    
 4   parent_id          46344 non-null   str    
 5   last_updated_at    203585 non-null  str    
 6   video_id           203585 non-null  str    
 7   likes              203585 non-null  int64  
 8   video_title        203585 non-null  str    
 9   video_date         203585 non-null  str    
 10  channel_id         203585 non-null  str    
 11  keyword_matched    203585 non-null  str    
 12  video_description  203585 non-null  str    
 13  char_count         203585 non-null  int64  
 14  word_count         203585 non-null  int64  
 15  avg_word_length    203585 non-null  float64
 16  has_url           

In [12]:
# Even if not null, some video descriptions, text or video_title might be empty strings. Let's check for that.
#filtered_data['video_description'].apply(lambda x: len(x.strip()) == 0).sum()
#filtered_data['text'].apply(lambda x: len(x.strip()) == 0).sum()
#filtered_data['video_title'].apply(lambda x: len(x.strip()) == 0).sum()
filtered_data = filtered_data[~(filtered_data['video_description'].apply(lambda x: len(x.strip()) == 0) | 
                                filtered_data['text'].apply(lambda x: len(x.strip()) == 0) | 
                                filtered_data['video_title'].apply(lambda x: len(x.strip()) == 0))]
filtered_data.shape

(203585, 34)

In [13]:
data = filtered_data.copy()

### 1.2. Standardizing Datetime

In [14]:
data['comment_date'] = pd.to_datetime(data['comment_date'])
data['video_date'] = pd.to_datetime(data['video_date'])
data['last_updated_at'] = pd.to_datetime(data['last_updated_at'])

In [15]:
# Sort by comment date for time series analysis
data = data.sort_values('comment_date').reset_index(drop=True)

print("Date columns converted:")
print(f"  Comment date range: {data['comment_date'].min()} to {data['comment_date'].max()}")
print(f"  Video date range: {data['video_date'].min()} to {data['video_date'].max()}")
print(f"  Total time span: {(data['comment_date'].max() - data['comment_date'].min()).days} days")

Date columns converted:
  Comment date range: 2021-12-10 03:58:59 to 2026-03-12 14:58:18
  Video date range: 2021-12-10 03:58:14 to 2026-03-12 12:01:37
  Total time span: 1553 days


## 2. Defining Features

### 2.1. Event Mention Detection

Based on Arc Raiders timeline and major announcements:
- **GenAI Voice Announcement**: When Embark Studios announced AI-generated voices
- **Business Model Change**: When they announced changing from free-to-play to paid
- **Game Launch**: Official release date

These dates will be used to calculate temporal features and decay functions.

In [16]:
MAJOR_EVENTS = {
    'game_announcement': pd.Timestamp('2021-12-10'),
    'genai_announcement': pd.Timestamp('2024-05-15'),
    'business_model_change': pd.Timestamp('2024-08-20'),
    'game_launch': pd.Timestamp('2025-10-30'),
    'early_access': pd.Timestamp('2025-11-15'),
}
print("Major Event Timeline:")
print("=" * 60)
for event, date in MAJOR_EVENTS.items():
    print(f"  {event:25s}: {date.strftime('%Y-%m-%d')}")
print("=" * 60)

Major Event Timeline:
  game_announcement        : 2021-12-10
  genai_announcement       : 2024-05-15
  business_model_change    : 2024-08-20
  game_launch              : 2025-10-30
  early_access             : 2025-11-15


Create binary features to detect if comments mention specific events.
This helps us identify which comments are directly responding to announcements.

In [17]:
EVENT_KEYWORDS = {
    'game_announcement': [
        'announcement', 'announced', 'reveal', 'revealed', 'teaser', 'teased',
        'first look', 'sneak peek', 'game announcement', 'game reveal'
    ],
    'genai': [
        'ai voice', 'ai-generated', 'artificial intelligence', 'generated voice',
        'ai audio', 'synthetic voice', 'ai narration', 'ai dialogue',
        'voice ai', 'ai-voiced', 'ai acting', 'machine voice'
    ],
    'business_model': [
        'free to play', 'f2p', 'freemium', 'paid game', 'pay to play', 'p2p',
        'price', 'cost', 'buying', 'purchase', 'buy the game', 'not free',
        'charging', 'monetization', 'business model', 'payment model'
    ],
    'launch': [
        'launch', 'release', 'released', 'launching', 'came out', 'available',
        'live', 'went live', 'early access'
    ]
}

In [18]:
def detect_event_mention(text, keywords):
    """Detect if text mentions any of the keywords"""
    if pd.isna(text):
        return False
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in keywords)

# Create event mention columns
data['mentions_genai'] = data['text'].apply(
    lambda x: detect_event_mention(x, EVENT_KEYWORDS['genai'])
)
data['mentions_business_model'] = data['text'].apply(
    lambda x: detect_event_mention(x, EVENT_KEYWORDS['business_model'])
)
data['mentions_launch'] = data['text'].apply(
    lambda x: detect_event_mention(x, EVENT_KEYWORDS['launch'])
)

In [19]:
# Summary of event mentions
print("Event Mention Detection Results:")
print("=" * 60)
print(f"Comments mentioning GenAI:           {data['mentions_genai'].sum():>8,} ({data['mentions_genai'].mean()*100:>5.2f}%)")
print(f"Comments mentioning Business Model:  {data['mentions_business_model'].sum():>8,} ({data['mentions_business_model'].mean()*100:>5.2f}%)")
print(f"Comments mentioning Launch:          {data['mentions_launch'].sum():>8,} ({data['mentions_launch'].mean()*100:>5.2f}%)")
print(f"Comments mentioning any event:       {(data['mentions_genai'] | data['mentions_business_model'] | data['mentions_launch']).sum():>8,}")
print("=" * 60)

Event Mention Detection Results:
Comments mentioning GenAI:                160 ( 0.08%)
Comments mentioning Business Model:     2,026 ( 1.00%)
Comments mentioning Launch:             3,824 ( 1.88%)
Comments mentioning any event:          5,860


### 2.2. Temporal Distance from Events

We calculate how many days each comment was posted relative to each major event.
This is crucial for measuring sentiment decay over time.

In [20]:
# Calculate days from each event
for event_name, event_date in MAJOR_EVENTS.items():
    col_name = f'days_from_{event_name}'
    data[col_name] = (data['comment_date'] - event_date).dt.days

In [21]:
# Display temporal features
print("Temporal Distance Features Created:")
print("=" * 40)
for event_name in MAJOR_EVENTS.keys():
    col_name = f'days_from_{event_name}'
    if col_name in data.columns:
        print(f"\n{event_name}:")
        print(f"  Min: {data[col_name].min():>6.0f} days (before event)")
        print(f"  Max: {data[col_name].max():>6.0f} days (after event)")
        print(f"  Comments before event: {(data[col_name] < 0).sum():>8,}")
        print(f"  Comments after event:  {(data[col_name] >= 0).sum():>8,}")

Temporal Distance Features Created:

game_announcement:
  Min:      0 days (before event)
  Max:   1553 days (after event)
  Comments before event:        0
  Comments after event:   203,585

genai_announcement:
  Min:   -887 days (before event)
  Max:    666 days (after event)
  Comments before event:   11,413
  Comments after event:   192,172

business_model_change:
  Min:   -984 days (before event)
  Max:    569 days (after event)
  Comments before event:   11,923
  Comments after event:   191,662

game_launch:
  Min:  -1420 days (before event)
  Max:    133 days (after event)
  Comments before event:   22,353
  Comments after event:   181,232

early_access:
  Min:  -1436 days (before event)
  Max:    117 days (after event)
  Comments before event:   22,731
  Comments after event:   180,854


### 2.3 Engagement Related Features

Here we create engagement-related features using like_count and other metrics.
These will be used for popularity-weighted sentiment scores.

In [22]:
data['has_likes'] = data['likes'] > 0
data['like_count_log'] = np.log1p(data['likes'])  # Log transform for skewed distribution

In [23]:
data['engagement_tier'] = pd.cut(
    data['likes'],
    bins=[-1, 0, 5, 20, 100, float('inf')],
    labels=['no_likes', 'low_engagement', 'medium_engagement', 'high_engagement', 'viral'])

In [24]:
data['comment_latency_days'] = (data['comment_date'] - data['video_date']).dt.total_seconds() / (24 * 3600)
data['comment_latency_hours'] = (data['comment_date'] - data['video_date']).dt.total_seconds() / 3600

In [25]:
data['latency_category'] = pd.cut(
    data['comment_latency_days'],
    bins=[-float('inf'), 0, 1, 7, 30, float('inf')],
    labels=['before_upload', 'same_day', 'within_week', 'within_month', 'after_month'])

In [26]:
print("Engagement Features Summary:")
print("=" * 40)
print(f"Comments with likes:        {data['has_likes'].sum():>8,} ({data['has_likes'].mean()*100:>5.2f}%)")
print(f"Mean like count:            {data['likes'].mean():>12.2f}")
print(f"Median like count:          {data['likes'].median():>12.2f}")
print(f"Max like count:             {data['likes'].max():>12.0f}")
print(f"\nEngagement Tier Distribution:")
for tier in data['engagement_tier'].cat.categories:
    count = (data['engagement_tier'] == tier).sum()
    print(f"  {tier:20s}: {count:>8,} ({count/len(data)*100:>5.2f}%)")

Engagement Features Summary:
Comments with likes:          57,406 (28.20%)
Mean like count:                    6.90
Median like count:                  0.00
Max like count:                    22393

Engagement Tier Distribution:
  no_likes            :  146,179 (71.80%)
  low_engagement      :   44,740 (21.98%)
  medium_engagement   :    7,354 ( 3.61%)
  high_engagement     :    3,461 ( 1.70%)
  viral               :    1,851 ( 0.91%)


## 3. Text Preprocessing

Preparing the text to make it suitable for sentiment analysis with RoBERTa model.

In [27]:
def preprocess_for_roberta(text):
    """
    Preprocess text for RoBERTa sentiment analysis.
    - Replace URLs with [URL] token
    - Replace @mentions with [USER] token
    - Remove excessive whitespace
    """
    if pd.isna(text):
        return ""
    
    # Replace URLs
    text = re.sub(r'http\S+|www\S+', '[URL]', text)
    
    # Replace @mentions
    text = re.sub(r'@\w+', '[USER]', text)
    
    # Clean whitespace
    text = ' '.join(text.split())
    
    return text

In [28]:
data['text_preprocessed'] = data['text'].apply(preprocess_for_roberta)

In [29]:
# Check preprocessing
print("Text Preprocessing Complete:")
print("=" * 60)
print(f"Original text with URLs:     {data['has_url'].sum():>6,}")
print(f"Original text with mentions: {data['has_mention'].sum():>6,}")
print(f"\nSample preprocessed texts:")
print("-" * 60)
for i, row in data[data['has_url'] | data['has_mention']].head(3).iterrows():
    print(f"\nOriginal:     {row['text'][:100]}...")
    print(f"Preprocessed: {row['text_preprocessed'][:100]}...")

Text Preprocessing Complete:
Original text with URLs:        264
Original text with mentions: 12,850

Sample preprocessed texts:
------------------------------------------------------------

Original:     @seanrobinson7748 thanks brother...
Preprocessed: [USER] thanks brother...

Original:     @ras6794 playing as a feMaLe is immersion breaking and cringey. So why do you care what I think?...
Preprocessed: [USER] playing as a feMaLe is immersion breaking and cringey. So why do you care what I think?...

Original:     @player-cw9nj Thanks...
Preprocessed: [USER]-cw9nj Thanks...


## 4. Adding context

Adding context from the video_description, as well as the top-level comment for replies, gives a **Conversational NLP** edge to my project. Even though it is not relevant for my RoBERTa-based model, it works great for the LLMs I am going to work with.

In [30]:
data['video_context'] = data['video_description'].str[:500]

In [31]:
# Creating the Parent Text Mapping first
text_map = data.set_index('comment_id')['text'].to_dict()

# Mapping parent texts to replies
data['parent_text'] = data['parent_id'].map(text_map)

## 5. Filtering the data for analysis

### 5.1 Keeping only the high quality data

In [32]:
# High-quality English comments only
data_clean = data[
    (data['language'] == 'en') &
    (data['word_count'] >= 3) &
    (~data['text'].isna())
].copy()

In [33]:
# Bot Removal (Deduplicate identical text)
data_clean = data_clean.drop_duplicates(subset=['text'], keep='first')
data_clean.info()

<class 'pandas.DataFrame'>
Index: 148569 entries, 0 to 203584
Data columns (total 51 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   comment_id                       148569 non-null  str           
 1   text                             148569 non-null  str           
 2   comment_date                     148569 non-null  datetime64[us]
 3   author_hash                      148569 non-null  str           
 4   parent_id                        31964 non-null   str           
 5   last_updated_at                  148569 non-null  datetime64[us]
 6   video_id                         148569 non-null  str           
 7   likes                            148569 non-null  int64         
 8   video_title                      148569 non-null  str           
 9   video_date                       148569 non-null  datetime64[us]
 10  channel_id                       148569 non-null  str       

### 5.2 Trimming the unnecessary columns from the dataset

In [34]:
column_list = data_clean.columns.tolist()
print(column_list)

['comment_id', 'text', 'comment_date', 'author_hash', 'parent_id', 'last_updated_at', 'video_id', 'likes', 'video_title', 'video_date', 'channel_id', 'keyword_matched', 'video_description', 'char_count', 'word_count', 'avg_word_length', 'has_url', 'has_mention', 'has_hashtag', 'exclamation_count', 'question_count', 'emoji_count', 'newline_count', 'uppercase_ratio', 'language', 'day_of_week', 'day_num', 'date', 'is_spike_x', 'z_score_x', 'is_spike_y', 'z_score_y', 'is_spike', 'z_score', 'mentions_genai', 'mentions_business_model', 'mentions_launch', 'days_from_game_announcement', 'days_from_genai_announcement', 'days_from_business_model_change', 'days_from_game_launch', 'days_from_early_access', 'has_likes', 'like_count_log', 'engagement_tier', 'comment_latency_days', 'comment_latency_hours', 'latency_category', 'text_preprocessed', 'video_context', 'parent_text']


In [38]:
columns_to_keep = [
    'comment_id', 'parent_id', 'parent_text',  # Conversation context
    'text_preprocessed',                       # Model input
    'video_context', 'video_id',               # Video context
    'comment_date', 'author_hash',             # Temporal/Identity
    'days_from_genai_announcement', 
    'days_from_business_model_change', 
    'days_from_game_announcement',
    'days_from_early_access',
    'days_from_game_launch',                   # Analysis variables
    'mentions_genai', 'mentions_business_model', 
    'likes', 'like_count_log',                 # Engagement
    'is_spike', 'z_score']                     # Spike detection

In [39]:
data_final = data_clean[columns_to_keep].copy()

In [40]:
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

data_final.to_csv(os.path.join(DATA_DIR, 'comments_for_analysis.csv'), index=False)
print(f"Dataset prepared with {len(data_final)} rows.")

Dataset prepared with 148569 rows.
